In [20]:
import numpy as np
import os
from pathlib import Path
from typing import Annotated, Union
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, StrictInt, StrictFloat, ConfigDict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

ImportError: cannot import name 'app' from 'app.main' (C:\Users\roudd\Documents\CreditRisk-Test\app\main.py)

## Créer l'API app/main.py

In [16]:
# Accéder au dossier du projet
os.chdir(r"C:\Users\roudd\Documents\CreditRisk-Test")

# Afficher le dossier courant
print("Dossier du projet :", os.getcwd())

# Vérifier les dossiers et fichiers existants
for dossier, sous_dossiers, fichiers in os.walk("."):
    print("\n", dossier)
    for fichier in fichiers:
        print("   -", fichier)

Dossier du projet : C:\Users\roudd\Documents\CreditRisk-Test

 .
   - .gitignore
   - README.md
   - requirements.txt
   - Tests_Prediction_Risque_Credit.ipynb

 .\.github

 .\.github\workflows
   - tests.yml

 .\.ipynb_checkpoints
   - Tests_Prediction_Risque_Credit-checkpoint.ipynb

 .\.pytest_cache
   - .gitignore
   - CACHEDIR.TAG
   - README.md

 .\.pytest_cache\v

 .\.pytest_cache\v\cache
   - nodeids

 .\.venv
   - .gitignore
   - pyvenv.cfg

 .\.venv\etc

 .\.venv\etc\jupyter

 .\.venv\etc\jupyter\jupyter_notebook_config.d
   - jupyterlab.json

 .\.venv\etc\jupyter\jupyter_server_config.d
   - jupyter-lsp-jupyter-server.json
   - jupyterlab.json
   - jupyter_server_terminals.json
   - notebook_shim.json

 .\.venv\Include

 .\.venv\Include\site

 .\.venv\Include\site\python3.14

 .\.venv\Include\site\python3.14\greenlet
   - greenlet.h

 .\.venv\Lib

 .\.venv\Lib\site-packages
   - a1_coverage.pth
   - brotli.py
   - configargparse.py
   - ipykernel_launcher.py
   - ipython_pygm

In [19]:

# ============================================================
# 1. Création et entraînement d'un modèle de test
# ============================================================

def creer_modele():

    rng = np.random.default_rng(42)

    nombre_observations = 1000

    age = rng.integers(18, 75, nombre_observations)
    revenu = rng.uniform(50000, 1000000, nombre_observations)
    montant = rng.uniform(100000, 5000000, nombre_observations)
    duree = rng.integers(3, 61, nombre_observations)
    anciennete = rng.uniform(0, 30, nombre_observations)

    X = np.column_stack([
        age,
        revenu,
        montant,
        duree,
        anciennete
    ])

    # Création d'une cible synthétique pour les tests
    risque = (
        0.8 * montant / revenu
        + 0.015 * duree
        - 0.04 * anciennete
        - 0.01 * age
    )

    probabilite = 1 / (1 + np.exp(-risque))

    y = (rng.random(nombre_observations) < probabilite).astype(int)

    modele = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(random_state=42))
    ])

    modele.fit(X, y)

    return modele


# Chargement du modèle
model = creer_modele()


# ============================================================
# 2. Création de l'application FastAPI
# ============================================================

app = FastAPI(
    title="CreditRisk API",
    description="API de prédiction du risque de défaut de crédit",
    version="1.0.0"
)


# ============================================================
# 3. Définition des données attendues
# ============================================================

class DemandeCredit(BaseModel):

    model_config = ConfigDict(extra="forbid")

    age: Annotated[StrictInt, Field(ge=18, le=100)]

    revenu_mensuel: Annotated[
        Union[StrictInt, StrictFloat],
        Field(gt=0, le=100_000_000)
    ]

    montant_credit: Annotated[
        Union[StrictInt, StrictFloat],
        Field(gt=0, le=100_000_000)
    ]

    duree_mois: Annotated[StrictInt, Field(ge=1, le=360)]

    anciennete_activite: Annotated[
        Union[StrictInt, StrictFloat],
        Field(ge=0, le=80)
    ]


# ============================================================
# 4. Endpoint de vérification de disponibilité
# ============================================================

@app.get("/health")
def health():

    return {
        "status": "ok",
        "model_loaded": model is not None
    }


# ============================================================
# 5. Endpoint de prédiction
# ============================================================

@app.post("/predict")
def predict(demande: DemandeCredit):

    if model is None:
        raise HTTPException(
            status_code=503,
            detail="Modèle de prédiction indisponible"
        )

    donnees = np.array([[
        demande.age,
        demande.revenu_mensuel,
        demande.montant_credit,
        demande.duree_mois,
        demande.anciennete_activite
    ]])

    probabilite = float(model.predict_proba(donnees)[0][1])

    return {
        "probabilite_defaut": probabilite,
        "decision": (
            "Risque élevé"
            if probabilite >= 0.5
            else "Risque faible"
        ),
        "version_modele": "1.0.0"
    }


# ============================================================
# 6. Endpoint des métadonnées du modèle
# ============================================================

@app.get("/model-info")
def model_info():

    return {
        "nom": "CreditRisk",
        "version": "1.0.0",
        "type": "Régression logistique",
        "model_loaded": model is not None
    }


print("API créée avec succès.")
print("Titre :", app.title)

for route in app.routes:
    if hasattr(route, "methods"):
        print(route.methods, route.path)

API créée avec succès.
Titre : CreditRisk API
{'GET', 'HEAD'} /openapi.json
{'GET', 'HEAD'} /docs
{'GET', 'HEAD'} /docs/oauth2-redirect
{'GET', 'HEAD'} /redoc
{'GET'} /health
{'POST'} /predict
{'GET'} /model-info


## Vérifier les réponses de l'API

In [21]:
from fastapi.testclient import TestClient
from app.main import app

# Créer un client de test
client = TestClient(app)

# 1. Tester GET /health
response_health = client.get("/health")

print("===== TEST HEALTH =====")
print("Code HTTP :", response_health.status_code)
print("Réponse :", response_health.json())


# 2. Tester GET /model-info
response_info = client.get("/model-info")

print("\n===== TEST MODEL-INFO =====")
print("Code HTTP :", response_info.status_code)
print("Réponse :", response_info.json())


# 3. Tester POST /predict avec une requête valide
donnees = {
    "age": 35,
    "revenu_mensuel": 250000,
    "montant_credit": 1000000,
    "duree_mois": 12,
    "anciennete_activite": 4
}

response_predict = client.post("/predict", json=donnees)

print("\n===== TEST PREDICT =====")
print("Code HTTP :", response_predict.status_code)
print("Réponse :", response_predict.json())

C:\Users\roudd\AppData\Local\Programs\Python\Python314\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


ImportError: cannot import name 'app' from 'app.main' (C:\Users\roudd\Documents\CreditRisk-Test\app\main.py)

In [22]:
from pathlib import Path

fichier = Path("app/main.py")

print("Fichier trouvé :", fichier.exists())
print("\nCONTENU DE app/main.py\n")
print(fichier.read_text(encoding="utf-8"))

Fichier trouvé : True

CONTENU DE app/main.py


